# ingest 计时：时间花在哪、按页外推要多久

对文件夹里按文件名排序的前 N 份 PDF 做**与 `run_folder.ipynb` 相同模式的完整 ingest**（ingest → requalify → qualify → index → publish，不答题），
用运行期 monkeypatch 给关键函数包一层计时（跑完恢复，不改任何源码），按"自身耗时"口径（扣除被包子调用）汇总到十几个桶，
所以各桶占比相加 = 100%，LLM 等待不会重复计入"版面"或"对象"。最后一格只打印约 20 行纯文本，方便手工抄录。

## 前置条件

同 `run_folder.ipynb`：项目根 `.env`（或真实环境变量）配好 `NB_PDF_DIR`、LLM（`OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`）、embedding（`OPENAI_EMBEDDING_MODEL`，或 `APP_EMBEDDING_*`）；
开头两格与 run_folder 相同：editable 安装（含 `pdf-onnx` = onnxruntime）+ Databricks 上重启解释器。**没有 LLM 配置时自动降级为离线模式**（首行 `mode=offline(no llm config)`：调用预算 0、token-hash 离线 embedding，只用于验证 notebook 本身能跑通，数字没有参考价值）。

## 可选变量

| 变量 | 默认 | 含义 |
|---|---|---|
| `NB_TIMING_MAX_PDFS` | 3 | 只测按文件名排序的前 N 份 PDF |
| `NB_TIMING_INGESTION_ROOT` | 全新临时目录 | 产物根；**要测冷启动就别复用**（复用会命中 model-cache 与阶段缓存，LLM 耗时失真）。临时目录在本地盘，若想测 run_folder 同款存储（如 Databricks 的 FUSE），指向 `APP_INGESTION_DIR` 下一个全新的子目录 |
| `NB_TIMING_PARALLEL` | 1 | 同时入库几份（对应 run_folder 的 `MAX_PARALLEL_DOCUMENTS`）；>1 时各桶秒数是跨线程累加，占比仍自洽，但"外推"按总桶秒数而非墙钟 |

产物只写 ingestion 根；PDF 目录只读。

In [ ]:
# 安装本仓库（editable）及一键流程需要的 extras：pdf = pdfspine，pdf-onnx = 本地 ONNX 版面 / 表格结构的
# onnxruntime 运行时（pdfspine[onnx]），service = 进程内评测用的 fastapi/httpx。
# 已经装过的环境可以跳过这一格。Databricks 上执行完要重启解释器：dbutils.library.restartPython()
%pip install -q -e "..[pdf,pdf-onnx,service]"

In [ ]:
import os

# 只在 Databricks 上重启 Python（让上一格 %pip 装的包生效）；其它环境什么都不做
# 本格是"notebook 不直接读环境变量"规则的例外：它在 import ragspine 之前执行，拿不到 get_settings()，只能直接读环境变量做平台探测
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()  # noqa: F821 - dbutils 由 Databricks 注入
else:
    print("非 Databricks 环境，跳过")

In [ ]:
# editable 安装靠 .pth 把 <repo>/src 加进 sys.path；Databricks 上该 .pth 不生效时显式补上（同 run_folder.ipynb）
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if (_src / "ragspine" / "__init__.py").is_file() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
print("Python", sys.version.split()[0], "| src =", _src)

In [ ]:
# ONNX 版面自检（ADR 0030 / 0039）：本地 PP-DocLayoutV3 版面能否用、权重在哪。onnxruntime 由安装格的
# pdf-onnx extra 装上；权重不随任何 wheel 分发：缺文件时这里从 ModelScope 下载版面权重（约 130MB）与表格结构权重 slanet-plus（约 8MB）——
# 配了 APP_ONNX_LAYOUT_MODEL 就下到那里，什么都没配就下到 <项目根>/data/models/pdfspine-onnx/（不进 git，之后自动找到）。
# 下载失败只打印提示、不中断，此时 LAYOUT_POLICY = "auto" 会退到 "deterministic-text-pages"（配置格会再打印原因）。
from enterprise_pdf_rag.adapters.onnx_partition import (
    ensure_onnx_layout_weights,
    onnx_layout_status,
)
from ragspine.common.evidence.configs import get_settings

print(ensure_onnx_layout_weights(get_settings().onnx_layout_model))
print(onnx_layout_status(get_settings().onnx_layout_model))
# 版面路由拿不准的页交给谁（ADR 0039）：model = 每页一次 VLM 版面（默认）；onnx-accept / text-only = 不调 VLM 版面
print("版面回退策略 APP_LAYOUT_FALLBACK =", get_settings().layout_fallback)

In [ ]:
# ───────────── 配置（与 run_folder.ipynb 同一套读取方式：项目根 .env / 真实环境变量）─────────────
import tempfile
from pathlib import Path

from pydantic_settings import BaseSettings, SettingsConfigDict

from enterprise_pdf_rag.adapters import folder_pipeline
from enterprise_pdf_rag.adapters.ingest_mode import (
    choose_layout_policy,
    choose_unverified_table_structure,
)
from ragspine.common.evidence.configs import ROOT_DIR, get_settings
from ragspine.common.evidence.providers.providers import load_llm_config

settings = get_settings()


class TimingSettings(BaseSettings):
    """本 notebook 独有的可选变量（Settings 不认识它们）；同样读项目根 .env，不向环境变量写入配置。"""

    model_config = SettingsConfigDict(
        env_file=ROOT_DIR / ".env", env_file_encoding="utf-8", extra="ignore"
    )
    nb_timing_max_pdfs: int = 3  # NB_TIMING_MAX_PDFS：只测按文件名排序的前 N 份
    nb_timing_ingestion_root: Path | None = None  # NB_TIMING_INGESTION_ROOT：产物根
    nb_timing_parallel: int = 1  # NB_TIMING_PARALLEL：同时入库几份


timing = TimingSettings()

# LLM 未配置 → 离线降级（只验证 notebook 能跑通；数字无参考价值）
try:
    load_llm_config()
    OFFLINE = False
except ValueError:
    OFFLINE = True
print("mode=offline(no llm config)" if OFFLINE else "mode=online")

PDF_DIR = settings.pdf_source_dir  # NB_PDF_DIR（只读）
MAX_PDFS = timing.nb_timing_max_pdfs
MAX_PARALLEL_DOCUMENTS = timing.nb_timing_parallel
# 默认一个全新临时目录，避免命中旧缓存。要测冷启动就别复用同一个目录：复用会让 model-cache / 阶段缓存命中，LLM 耗时失真。
INGESTION_ROOT = timing.nb_timing_ingestion_root or Path(tempfile.mkdtemp(prefix="ingest_timing_"))
INGESTION_ROOT = INGESTION_ROOT.expanduser().resolve()

# ---- 以下与 run_folder.ipynb 当前实际使用的一致（与 run_folder 对齐）----
INGEST_MODE = "lite"  # 与 run_folder 对齐
BUILD_TREE = None  # 与 run_folder 对齐：None = 跟随模式（lite 不建 tree）
LAYOUT_POLICY = "auto"  # 与 run_folder 对齐
UNVERIFIED_TABLE_STRUCTURE = "auto"  # 与 run_folder 对齐
REQUALIFY = True  # 与 run_folder 对齐
MAX_LIVE_CALLS_PER_PDF = (
    0 if OFFLINE else "auto"
)  # 与 run_folder 对齐（"auto"）；离线降级时为 0 = 不发任何调用
MAX_LIVE_CALLS_TOTAL = None  # 与 run_folder 对齐

EFFECTIVE_LAYOUT, LAYOUT_REASON = choose_layout_policy(
    LAYOUT_POLICY, ingest_mode=INGEST_MODE, onnx_layout_model=settings.onnx_layout_model
)
EFFECTIVE_TABLE_STRUCTURE, TABLE_STRUCTURE_REASON = choose_unverified_table_structure(
    UNVERIFIED_TABLE_STRUCTURE, ingest_mode=INGEST_MODE
)

# ---- 校验（先于任何写入 / 模型调用）----
if PDF_DIR is None:
    raise RuntimeError("未设置 PDF 文件夹：请在项目根 .env 里设 NB_PDF_DIR")
PDF_DIR = Path(PDF_DIR).expanduser().resolve()
if not PDF_DIR.is_dir():
    raise RuntimeError(f"PDF 文件夹不存在：{PDF_DIR}")
if MAX_PDFS < 1 or MAX_PARALLEL_DOCUMENTS < 1:
    raise RuntimeError("NB_TIMING_MAX_PDFS / NB_TIMING_PARALLEL 必须 >= 1")
if INGESTION_ROOT.is_relative_to(PDF_DIR) or PDF_DIR.is_relative_to(INGESTION_ROOT):
    raise RuntimeError(
        f"ingestion 根 {INGESTION_ROOT} 与 PDF 源目录 {PDF_DIR} 重叠：PDF 目录是只读的"
    )
SELECTED_PDFS = folder_pipeline.discover_pdfs(PDF_DIR)[:MAX_PDFS]
if not SELECTED_PDFS:
    raise RuntimeError(f"{PDF_DIR} 下没有 PDF")

print("PDF 源目录 :", PDF_DIR, f"（本次测前 {len(SELECTED_PDFS)} 份）")
print(
    "ingestion  :",
    INGESTION_ROOT,
    "（自定义）" if timing.nb_timing_ingestion_root else "（全新临时目录）",
)
print("入库模式   :", INGEST_MODE, "| 版面:", EFFECTIVE_LAYOUT, f"（{LAYOUT_REASON}）")
print("表格结构   :", EFFECTIVE_TABLE_STRUCTURE, f"（{TABLE_STRUCTURE_REASON}）")
print("并发入库   :", MAX_PARALLEL_DOCUMENTS, "| 调用预算/PDF:", MAX_LIVE_CALLS_PER_PDF)

In [ ]:
# ───────────── 计时器：自身耗时口径（每个被包函数只记扣除了被包子调用之后的时间，所以各桶相加 = 总计）─────────────
import functools
import importlib
import sqlite3
import sys
import threading
import time
from collections import defaultdict
from contextlib import contextmanager

# 桶的顺序就是最后一格的行序。"(自身)" = 不含子桶：LLM 等待只计入对应 vlm 桶，不会重复计入 layout / objects。
BUCKETS = (
    "pdfspine_extract",  # extract_document / get_svg_image（含图 SVG 解析、源绘制轨迹读取）
    "layout_local",  # 确定性文字页 / ONNX 版面分区（含 find_tables）
    "svg_png",  # SVG→PNG（resvg）
    "layout_vlm",  # 版面 VLM：task page-layout-*（含模型分区器自身的准备）
    "obj_ir_vlm",  # 图表 IR / 视觉 IR 的 VLM：task chart-branch-* / visual-ir-*
    "obj_desc_vlm",  # 对象描述 VLM：task description-branch-* / visual-description-*
    "objects_local",  # semantic_objects.process 自身 + 图表映射 + 表格抽取 / TSR 推理（不含上面的 VLM 与 resvg）
    "page_meta",  # 页元数据提取（含其 LLM 调用：task page-metadata-*）
    "embedding",  # provider 的 embed_*（次数 = 调用数，向量条数另计）
    "qual_index_pub",  # requalify / qualify / index / publish / tree（自身，含 tree 的 LLM）
    "store_write",  # LocalDocumentStore / ProcessingStore / 对象后端 / model-cache 的写与校验（不含 sqlite 与哈希）
    "sqlite",  # sqlite3 connection.execute / executemany / executescript / commit
    "hash",  # hashlib.sha256 构造（含对整块字节求哈希）
    "export_review",  # export_document_review / export_processing_review
    "other",  # _run_document 里没被上面任何桶覆盖的部分
)


class Prof:
    def __init__(self):
        self.tls = threading.local()
        self.lock = threading.Lock()
        self.on = False
        self.secs = defaultdict(float)
        self.calls = defaultdict(int)
        self.count = defaultdict(int)  # fallback_pages / embed_vectors
        self.tasks = defaultdict(lambda: [0, 0, 0.0])  # task -> [发出, model-cache 命中, 墙钟秒]

    @contextmanager
    def span(self, bucket, counted=True):
        stack = self.tls.__dict__.setdefault("stack", [])
        frame = [time.perf_counter(), 0.0]  # 开始时刻, 子 span 累计秒
        stack.append(frame)
        try:
            yield
        finally:
            elapsed = time.perf_counter() - frame[0]
            stack.pop()
            if stack:
                stack[-1][1] += elapsed
            if self.on:
                with self.lock:
                    self.secs[bucket] += elapsed - frame[1]
                    if counted:
                        self.calls[bucket] += 1

    def add(self, name, value=1):
        if self.on:
            with self.lock:
                self.count[name] += value


prof = Prof()
_PATCHED = []  # (owner, 属性名, 原对象)，跑完恢复
_SKIPPED = []  # 当前版本找不到的目标，只提示


def _llm_bucket(task):
    if task.startswith("page-layout"):
        return "layout_vlm"
    if task.startswith("page-metadata"):
        return "page_meta"
    if "tree" in task:
        return "qual_index_pub"
    if "description" in task:
        return "obj_desc_vlm"
    return "obj_ir_vlm"


def _wrap_llm(original):
    # JsonCompletionClient.complete_json / complete_text_json 是所有入库模型调用的唯一入口；task 参数区分用途
    @functools.wraps(original)
    def inner(self_, *args, **kwargs):
        task = str(kwargs.get("task", "?"))
        bucket, hit, started = _llm_bucket(task), False, time.perf_counter()
        try:
            with prof.span(bucket, counted=False):
                result = original(self_, *args, **kwargs)
            hit = bool(result.cache_hit)
            return result
        finally:
            if prof.on:
                with prof.lock:
                    record = prof.tasks[task]
                    record[1 if hit else 0] += 1
                    record[2] += time.perf_counter() - started
                    prof.calls[bucket] += 1

    return inner


def _install(target, bucket, *, counted=True, before=None, llm=False):
    # target = "包.模块:函数" 或 "包.模块:类.方法"；模块级函数还会替换所有 `from x import f` 出来的同名绑定
    module_name, _, qual = target.partition(":")
    module = importlib.import_module(module_name)
    owner, name = module, qual
    if "." in qual:
        class_name, name = qual.split(".")
        owner = getattr(module, class_name, None)
    original = None if owner is None else vars(owner).get(name)
    if (
        original is None
        or not callable(original)
        or isinstance(original, (staticmethod, classmethod))
    ):
        _SKIPPED.append(target)
        return

    if llm:
        wrapper = _wrap_llm(original)
    else:

        @functools.wraps(original)
        def wrapper(*args, **kwargs):
            if before is not None:
                before(args, kwargs)
            with prof.span(bucket, counted):
                return original(*args, **kwargs)

    owners = [owner]
    if owner is module:
        owners = [
            mod
            for mod_name, mod in list(sys.modules.items())
            if mod is not None
            and (mod is module or mod_name.startswith(("enterprise_pdf_rag", "ragspine")))
            and getattr(mod, "__dict__", {}).get(name) is original
        ]
    for each in owners:
        setattr(each, name, wrapper)
        _PATCHED.append((each, name, original))


class _TimedConnection(sqlite3.Connection):
    def execute(self, *args, **kwargs):
        with prof.span("sqlite"):
            return super().execute(*args, **kwargs)

    def executemany(self, *args, **kwargs):
        with prof.span("sqlite"):
            return super().executemany(*args, **kwargs)

    def executescript(self, *args, **kwargs):
        with prof.span("sqlite"):
            return super().executescript(*args, **kwargs)

    def commit(self):
        with prof.span("sqlite", counted=False):
            return super().commit()


def restore_patches():
    while _PATCHED:
        owner, name, original = _PATCHED.pop()
        setattr(owner, name, original)


_A = "enterprise_pdf_rag.adapters."
_OB = "ragspine.common.evidence.object_backend."
_LM = "ragspine.common.evidence.providers.local_models:LocalEmbeddingAdapter."


def install_patches():
    for target, bucket in (
        (_A + "pdfspine_document:PdfspineDocumentAdapter.extract_document", "pdfspine_extract"),
        (_A + "pdfspine_figure:PdfspineFigureParser.extract", "pdfspine_extract"),
        (_A + "source_paint:_read_source_trace", "pdfspine_extract"),
        (_A + "deterministic_partition:TextPageRouterPartitioner.partition", "layout_local"),
        (_A + "deterministic_partition:TextPageRouterPartitioner._tables", "layout_local"),
        (_A + "onnx_partition:OnnxPagePartitioner.partition", "layout_local"),
        (_A + "figure_reasoning:render_svg_png", "svg_png"),
        (_A + "semantic_objects:SemanticObjectAdapter.process", "objects_local"),
        (_A + "page_metadata_extraction:annotate_page_metadata", "page_meta"),
        (_A + "draft_publication:qualify_draft", "qual_index_pub"),
        (_A + "draft_publication:index_draft", "qual_index_pub"),
        (_A + "draft_publication:publish_draft", "qual_index_pub"),
        (_A + "visual_requalification:requalify_visual_objects", "qual_index_pub"),
        (_A + "document_tree_extraction:annotate_document_tree", "qual_index_pub"),
        (_A + "pdf_ingestion:_export_review", "export_review"),
        (_A + "pdf_ingestion:export_document_review", "export_review"),
        (_A + "processing_store:ProcessingStore.cache", "store_write"),
        (_A + "processing_store:ProcessingStore.cache_output", "store_write"),
        (_A + "processing_store:ProcessingStore.save_draft", "store_write"),
        (_A + "processing_store:ProcessingStore.publish", "store_write"),
        (_A + "processing_store:ProcessingStore.load", "store_write"),  # 读 manifest 时的校验
        (_A + "document_store:LocalDocumentStore.put", "store_write"),
        (_A + "document_store:LocalDocumentStore.publish", "store_write"),
        (_A + "document_store:LocalDocumentStore.verify_snapshot", "store_write"),
    ):
        _install(target, bucket)
    # 这些是上面某个桶的子步骤：自身耗时并入同一个桶（或各自的桶），不单独计次数
    for target, bucket in (
        (_A + "visual_semantics:_prepare", "objects_local"),
        (_A + "visual_semantics:VisualSemanticAdapter.infer", "objects_local"),
        (_A + "chart_semantics:ModelChartExtractor.infer", "objects_local"),
        (_A + "chart_semantics:ModelDescriptionGenerator.infer", "objects_local"),
        (_A + "chart_semantics:map_chart", "objects_local"),
        (_A + "pdfspine_tables:PdfspineTableAdapter.extract", "objects_local"),
        (_A + "pdfspine_tsr:SlanetPlusRecognizer.recognize", "objects_local"),
        (_A + "pdfspine_tsr:infer_table_grid", "objects_local"),
        (_A + "processing_export:export_processing_review", "export_review"),
        (_A + "document_store:LocalDocumentStore.verify", "store_write"),
        (_OB + "files:FileBackend.put_object", "store_write"),
        (_OB + "files:FileBackend.put_stage_entry", "store_write"),
        (_OB + "files:FileBackend.put_record", "store_write"),
        (_OB + "files:FileBackend.set_pointer", "store_write"),
        (_OB + "files:FileBackend.verify_many", "store_write"),
        (_OB + "files:FileModelCacheBackend.put_record", "store_write"),
        (_OB + "files:FileModelCacheBackend.put_response", "store_write"),
        (_OB + "files:FileModelCacheBackend.put_context", "store_write"),
        (_OB + "sqlite:SqliteBackend.put_object", "store_write"),
        (_OB + "sqlite:SqliteBackend.put_stage_entry", "store_write"),
        (_OB + "sqlite:SqliteBackend.put_record", "store_write"),
        (_OB + "sqlite:SqliteBackend.verify_many", "store_write"),
        (_OB + "sqlite:SqliteModelCacheBackend.put_record", "store_write"),
        (_OB + "sqlite:SqliteModelCacheBackend.put_response", "store_write"),
        (_OB + "sqlite:SqliteModelCacheBackend.put_context", "store_write"),
    ):
        _install(target, bucket, counted=False)
    # 模型分区器被调用 = 该页回退到模型版面（model 策略下则每页都是）
    _install(
        _A + "page_partition:ModelPagePartitioner.partition",
        "layout_vlm",
        counted=False,
        before=lambda args, kwargs: prof.add("fallback_pages"),
    )
    # embedding：次数 = provider 调用数；向量条数 = 单条算 1，批量算 len(texts)
    for owner_path in (_LM, _A + "offline:OfflineDescriptionEmbedder."):
        for method in ("embed_description", "embed_query"):
            _install(
                owner_path + method,
                "embedding",
                before=lambda args, kwargs: prof.add("embed_vectors"),
            )
        _install(
            owner_path + "embed_descriptions",
            "embedding",
            before=lambda args, kwargs: prof.add("embed_vectors", len(args[1])),
        )
    # 哈希：hashlib.sha256 是 C 函数，构造时对传入字节求哈希；其余 `from hashlib import sha256` 的绑定一并替换
    _install("hashlib:sha256", "hash")
    _install(_A + "folder_pipeline:_hash_file", "hash", counted=False)
    # 所有入库 LLM / VLM 调用的唯一入口
    jc = "ragspine.common.evidence.providers.json_completion:JsonCompletionClient."
    _install(jc + "complete_json", "", llm=True)
    _install(jc + "complete_text_json", "", llm=True)
    # 文档级根 span：其自身耗时 = other
    _install(_A + "folder_pipeline:_run_document", "other")
    # sqlite：给每个新连接装计时的 Connection 子类
    _real_connect = sqlite3.connect

    def connect(*args, **kwargs):
        kwargs.setdefault("factory", _TimedConnection)
        return _real_connect(*args, **kwargs)

    sqlite3.connect = connect
    _PATCHED.append((sqlite3, "connect", _real_connect))


print("计时器已定义（尚未安装 patch）；桶:", ", ".join(BUCKETS))

In [ ]:
# ───────────── 跑：安装计时 → run_folder_pipeline（只 ingest，不答题）→ 恢复 ─────────────
import os

from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline

_T0 = time.perf_counter()
PDF_BYTES = sum(pdf.stat().st_size for pdf in SELECTED_PDFS)


def progress(event, payload):
    # 只打印文档的开始 / 结束，页级进度不打印
    if event in ("document_start", "document_done"):
        brief = {
            k: v
            for k, v in payload.items()
            if k in ("pdf", "status", "error", "failed_stage", "pages")
        }
        if "pdf" in brief:
            brief["pdf"] = Path(brief["pdf"]).name
        print(f"[{time.perf_counter() - _T0:7.1f}s]", event, brief)


# 只读 patch：只测前 MAX_PDFS 份；忽略 .env 里的题集 / NB_REPORT_DIR，不答题、不写 report
_orig_discover = folder_pipeline.discover_pdfs
_orig_get_settings = folder_pipeline.get_settings
folder_pipeline.discover_pdfs = lambda folder: _orig_discover(folder)[:MAX_PDFS]
folder_pipeline.get_settings = lambda: _orig_get_settings().model_copy(
    update={"questions_path": None, "report_dir": None}
)

EMBEDDER = None  # 在线：run_folder 同款（settings 里的 embedding 配置，开跑前探测一次）
if OFFLINE:
    # 离线降级：占位 LLM 配置（预算 0，不会发出任何请求）+ token-hash 离线 embedding；仅为验证 notebook 本身能跑通
    from enterprise_pdf_rag.adapters.offline import OfflineDescriptionEmbedder

    _offline_config = load_llm_config(
        {
            "APP_LLM_API_KEY": "offline-0123456789abcdef",
            "APP_LLM_BASE_URL": "https://offline.invalid/v1",
            "APP_LLM_MODEL": "offline-model",
        }
    )
    for _name, _module in list(sys.modules.items()):
        if (
            _module is not None
            and _name.startswith(("enterprise_pdf_rag", "ragspine"))
            and getattr(_module, "__dict__", {}).get("load_llm_config") is load_llm_config
        ):
            _module.load_llm_config = lambda environment=None: _offline_config
            _PATCHED.append((_module, "load_llm_config", load_llm_config))
    EMBEDDER = OfflineDescriptionEmbedder()

install_patches()
print(f"已包装 {len(_PATCHED)} 处绑定；当前版本找不到的目标 {len(_SKIPPED)} 个：{_SKIPPED}")
prof.on = True
try:
    result = run_folder_pipeline(
        PDF_DIR,
        questions=None,
        ingestion_root=INGESTION_ROOT,
        max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
        max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
        max_parallel_documents=MAX_PARALLEL_DOCUMENTS,
        ingest_mode=INGEST_MODE,
        build_tree=BUILD_TREE,
        unverified_table_structure=EFFECTIVE_TABLE_STRUCTURE,
        layout_policy=EFFECTIVE_LAYOUT,
        requalify=REQUALIFY,
        embedder=EMBEDDER,
        report_dir=None,
        progress=progress,
    )
finally:
    prof.on = False
    WALL_S = time.perf_counter() - _T0
    restore_patches()
    folder_pipeline.discover_pdfs = _orig_discover
    folder_pipeline.get_settings = _orig_get_settings

# 产物目录大小与文件数（整个 ingestion 根；os.walk 跨平台）
OUT_BYTES = OUT_FILES = 0
for _dir, _dirs, _files in os.walk(INGESTION_ROOT):
    for _file in _files:
        try:
            OUT_BYTES += os.lstat(os.path.join(_dir, _file)).st_size
            OUT_FILES += 1
        except OSError:
            pass

DOCS = [d for d in result.documents if d.status not in ("duplicate_of", "skipped_not_referenced")]
PAGES = sum(len(d.ingestion.selected_physical_pages) for d in DOCS if d.ingestion is not None)
print(f"完成：wall {WALL_S:.1f}s | ok={result.ok} | live_calls={result.live_calls.model_dump()}")

In [ ]:
# ───────────── 明细（可不抄）：每份 PDF 与每个 LLM task ─────────────
for d in DOCS:
    pages = (
        "-"
        if d.ingestion is None
        else f"{d.ingestion.pages_complete}/{len(d.ingestion.selected_physical_pages)}"
    )
    print(
        f"{Path(d.pdf_path).name[:40]:<40} status={d.status} pages={pages} elapsed={d.elapsed_s:.1f}s size={Path(d.pdf_path).stat().st_size / 1e6:.1f}MB {d.error or ''}"[
            :200
        ]
    )
for task, (sent, hits, secs) in sorted(prof.tasks.items(), key=lambda kv: -kv[1][2]):
    print(f"llm task {task:<34} sent={sent} cache_hit={hits} s={secs:.1f}")
if not prof.tasks:
    print("llm task: 本次没有任何模型调用")

In [ ]:
# ───────────── 汇总：约 20 行纯文本，供手工抄录 ─────────────
total = sum(prof.secs.values()) or 1e-9
pages = max(PAGES, 1)
sent = sum(v[0] for v in prof.tasks.values())
hits = sum(v[1] for v in prof.tasks.values())
llm_s = sum(v[2] for v in prof.tasks.values())
spp = (WALL_S if MAX_PARALLEL_DOCUMENTS == 1 else total) / pages  # 并发时按总桶秒数（跨线程累加）
lines = []
if OFFLINE:
    lines.append("mode=offline(no llm config)")
lines.append(
    f"ingest_timing v1 | pdfs={len(DOCS)} pages={PAGES} wall={WALL_S:.1f}s s/page={WALL_S / pages:.2f} "
    f"parallel={MAX_PARALLEL_DOCUMENTS} ok={sum(d.status == 'published' for d in DOCS)}/{len(DOCS)}"
)
lines.append(
    f"fallback_pages={prof.count['fallback_pages']}/{PAGES} llm_calls={sent} llm_s={llm_s:.1f} "
    f"cache_hits={hits} embed_vectors={prof.count['embed_vectors']}"
)
lines.append(f"{'stage(self)':<16}{'s':>8}{'%':>7}{'s/page':>8}{'n':>7}")
for bucket in BUCKETS:
    secs = prof.secs[bucket]
    lines.append(
        f"{bucket:<16}{secs:>8.1f}{100 * secs / total:>7.1f}{secs / pages:>8.2f}{prof.calls[bucket]:>7d}"
    )
lines.append(
    f"disk: pdf={PDF_BYTES / 1e6:.1f}MB out={OUT_BYTES / 1e6:.1f}MB files={OUT_FILES} per_page={OUT_BYTES / 1e6 / pages:.1f}MB"
)
serial_h = spp * 50000 / 3600
lines.append(
    f"extrapolate 50000 pages: serial={serial_h:.1f}h parallel8={serial_h / 8:.1f}h(linear assumption)"
)
print("\n".join(lines))